In [40]:
import os
import json
import shutil
from dotenv import load_dotenv
from langchain_community.document_loaders import TextLoader
from langchain_openai import (
    OpenAIEmbeddings,
    OpenAI
)
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document
from pathlib import Path
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import (
    EnsembleRetriever,
    ContextualCompressionRetriever
)

from langchain_classic.retrievers.document_compressors import LLMChainExtractor

load_dotenv(override=True)



True

In [12]:
PROJECT_ROOT = Path(os.getcwd()).resolve().parents[0]
EMBEDDING_MODEL_SMALL = "text-embedding-3-small"
CHROMA_PERSIST_DIR = str(PROJECT_ROOT / "data" / "chroma_db")
COLLECTION_NAME = "jd_intelligence_baseline_500d"

In [13]:
db_path = PROJECT_ROOT / "data" / "chroma_db"

if db_path.exists():
    if db_path.is_dir():
        shutil.rmtree(db_path)  
        print("Directory deleted successfully.")
    else:
        os.remove(db_path)     
else:
    print("Path does not exist.")

Path does not exist.


In [19]:
with open(PROJECT_ROOT / "checkpoint.json", 'r', encoding='utf-8') as f:
    checkpoint = json.load(f)

checkpoint

[{'ingested_raw_filenames': [], 'normalized_output_filenames': []}]

## **1. Document Loader**

In [20]:
def load_and_normalize_jd(document_path: Path) -> str:

    loader = TextLoader(document_path)
    documents = loader.load()

    text_data = documents[0].page_content
    # Fixed Syntax
    data = [line for line in text_data.split("\n\n") if line != '\n']

    final_text = []

    for block in data:
        block = block.strip()
        if "\n" in block:
            # Split the sub-lines inside this block
            sub_lines = block.split("\n")
            # Join them back with a single SPACE, filtering out empty entries
            block = ' '.join([sl.strip() for sl in sub_lines if sl.strip()])

        final_text.append(block)

    return ' '.join(final_text)


In [21]:


directory_path = PROJECT_ROOT / "data" / "raw_jds"
normalized_files = []

for file in os.listdir(directory_path):
    # Track files using the clean string filename ('visa_de.txt')
    if file not in checkpoint[0]["ingested_raw_filenames"]:
        full_file_path = directory_path / file
        
        if full_file_path.is_file():
            normalized_files.append(file)
            print(f"Successfully processed: {file}")

            # Extract, clean, and write the text file
            cleaned_data = load_and_normalize_jd(full_file_path)
            output_path = PROJECT_ROOT / "data" / "cleaned_jds" / f"{full_file_path.stem}.txt"
            
            with open(output_path, "w", encoding="utf-8") as f:
                f.write(cleaned_data)
            
            # Appending the text filename string so JSON can save it safely
            checkpoint[0]["ingested_raw_filenames"].append(file)

print(f"\nTotal new files processed normalized: {len(normalized_files)}")

with open(PROJECT_ROOT / "checkpoint.json", 'w', encoding='utf-8') as f:
    json.dump(checkpoint, f, indent=4)

print("Checkpoint database updated cleanly.")

Successfully processed: capgmini_javadeveloper.txt
Successfully processed: hcl_tech_java_developer.txt
Successfully processed: glballogic_javadeveloper.txt
Successfully processed: tiger_analytics_de.txt
Successfully processed: innodata_india_de.txt
Successfully processed: tcs_gcp_de.txt
Successfully processed: visa_de.txt
Successfully processed: accenture_data_platform_de.txt

Total new files processed normalized: 8
Checkpoint database updated cleanly.


## **2. Text Splitter, Embeddings and Vector Store**

In [22]:
## Text Splitteing or Chunking
def chunk_jd_text(splitter:RecursiveCharacterTextSplitter, source_path:Path) ->  list[Document]:

    cleaned_text = ""

    with open(source_path, 'r', encoding="utf-8") as f:
        cleaned_text = f.read()

    chunks = splitter.create_documents(
        texts=[cleaned_text],
        metadatas=[
            {
                "source" : source_path.stem,
                "Description" : f"{(source_path.stem).split("_")[0]} {(source_path.stem).split("_")[1]} job description"
            }
        ]
    )
    
    return chunks
    
    

In [23]:

cleaned_file_path = Path(PROJECT_ROOT / "data" / "cleaned_jds")

## 1. Text Splitter configuration
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", " ", ""]
)

## 2. Low-dimension Embedding configuration
embeddings_engine = OpenAIEmbeddings(
    model=EMBEDDING_MODEL_SMALL,
    dimensions=500,  # Your target evaluation baseline length
    max_retries=3
)

## 3. Persistent Storage Handler connection
vector_store = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embeddings_engine,
    persist_directory=CHROMA_PERSIST_DIR
)

files_processed_this_run = 0

all_documents = []

for file in os.listdir(cleaned_file_path):
    if not file.startswith('.') and file.endswith('.txt'):
        
        if file not in checkpoint[0]["normalized_output_filenames"]:
            cleaned_full_file_path = cleaned_file_path / file
            
            # Extract live documents into memory
            chunked_data = chunk_jd_text(splitter=splitter, source_path=cleaned_full_file_path)
            
            if chunked_data:
                # Core fix: add_documents pushes directly to DB and returns a list of chunk IDs
                chunk_ids = vector_store.add_documents(chunked_data)
                all_documents.extend(chunked_data)
                files_processed_this_run += 1

                print(f"==================================================")
                print(f"SUCCESS: {file}")
                print(f" -> Generated Chunks: {len(chunked_data)}")
                print(f" -> Indexed Chunk IDs: {len(chunk_ids)} registered keys")
                print(f" -> Sample ID range: {chunk_ids[0]} ... {chunk_ids[-1]}")
                print(f"==================================================\n")

            # Mark the file as processed in memory state
            checkpoint[0]['normalized_output_filenames'].append(file)

print(f"Processing Batch Complete! {files_processed_this_run} new files indexed safely.")

# 4. Commit tracking records to disk
with open(PROJECT_ROOT / "checkpoint.json", 'w', encoding='utf-8') as f:
    json.dump(checkpoint, f, indent=4)

print("Checkpoint ledger committed safely to checkpoint.json.")

/var/folders/gp/0r9x4vj97xn9cbjqk5n01ykc0000gn/T/ipykernel_71936/26625458.py:18: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vector_store = Chroma(


SUCCESS: capgmini_javadeveloper.txt
 -> Generated Chunks: 2
 -> Indexed Chunk IDs: 2 registered keys
 -> Sample ID range: ab5201ba-75aa-4a1b-a181-9fc407b57f85 ... d805aa43-f836-4192-bca3-0ab2a7f900e0

SUCCESS: hcl_tech_java_developer.txt
 -> Generated Chunks: 6
 -> Indexed Chunk IDs: 6 registered keys
 -> Sample ID range: 025550db-524d-402c-be16-0dec313293d1 ... efa1899a-7c47-4dbd-b92a-7ed937828c24

SUCCESS: glballogic_javadeveloper.txt
 -> Generated Chunks: 7
 -> Indexed Chunk IDs: 7 registered keys
 -> Sample ID range: 18903924-0622-40ac-a967-48c9c3d8ffc7 ... 538c3b1b-82df-4ee3-ad1f-01af52737a30

SUCCESS: tiger_analytics_de.txt
 -> Generated Chunks: 11
 -> Indexed Chunk IDs: 11 registered keys
 -> Sample ID range: a02a7102-6119-4439-bbb4-8c5333b317f5 ... da4594ce-7563-4f90-a4be-2437b83a72ed

SUCCESS: innodata_india_de.txt
 -> Generated Chunks: 5
 -> Indexed Chunk IDs: 5 registered keys
 -> Sample ID range: 19e9d78e-b1e9-40b0-966b-3d16ffabdf1d ... 06054a55-f630-428a-8d24-ffd8acf8ce00


-- not required right now

In [24]:
## Retrival
## 1. Similarity Search
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 2
    }
)

query = "Looking for a GCP data engineer Role without Spark or Databricks as a skill"

results_with_scores = vector_store.similarity_search_with_score(query)

print("--- Similarity Search Results with L2 Distance Scores ---")
for i, (doc, score) in enumerate(results_with_scores):
    print(f"\n[Match {i}] Source File: {doc.metadata.get('source')}.txt")
    print(f"    Distance Score: {score:.4f}")  # Smaller = Closer/More accurate match
    print(f"    Content Snippet: {doc.page_content[:150]}...")

--- Similarity Search Results with L2 Distance Scores ---

[Match 0] Source File: innodata_india_de.txt
    Distance Score: 0.5373
    Content Snippet: Job description Role- GCP Data Engineer Responsibilities: • Design and implement data-driven solutions on GCP including BigQuery, Cloud Storage, Dataf...

[Match 1] Source File: tcs_gcp_de.txt
    Distance Score: 0.6690
    Content Snippet: Job description TCS Hiring for Digital : GCP Data Engineer - BigQuery Role!! TCS presents an excellent opportunity forGCP Data Engineer - BigQuery Des...

[Match 2] Source File: tcs_gcp_de.txt
    Distance Score: 0.6846
    Content Snippet: in programming languages such as Python, PySpark data manipulation and engineering tasks. •Expertise in SQL and NoSQL databases •In-depth knowledge of...

[Match 3] Source File: innodata_india_de.txt
    Distance Score: 0.6872
    Content Snippet: and pipeline reliability. Skills Required : • Strong hands-on expertise with GCP services: BigQuery, Dataflow, Pub/Sub

In [25]:
## Similarity Search with Thershold

threshold = 0.5

retriever_thershould = vector_store.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"score_threshold": threshold},
)

results = retriever_thershould.invoke(query)

print(f"=== Threshold = {threshold} — {len(results)} document(s) returned ===")
for i, doc in enumerate(results, 1):
    print(f"  [{i}] metadata={doc.metadata}: {doc.page_content}")


=== Threshold = 0.5 — 4 document(s) returned ===
  [1] metadata={'source': 'innodata_india_de', 'Description': 'innodata india job description'}: Job description Role- GCP Data Engineer Responsibilities: • Design and implement data-driven solutions on GCP including BigQuery, Cloud Storage, Dataflow, Pub/Sub, and Looker/BI. • Build ETL scripts using SQL and Python to extract, clean, and transform structured and unstructured data from ERP, procurement, logistics, and facility management systems. • Develop and optimize data pipelines for ingestion, transformation, and loading into enterprise data lakes and warehouses. • Build and extend
  [2] metadata={'Description': 'tcs gcp job description', 'source': 'tcs_gcp_de'}: Job description TCS Hiring for Digital : GCP Data Engineer - BigQuery Role!! TCS presents an excellent opportunity forGCP Data Engineer - BigQuery Desired Experience Range: 5-10 years Location: CHENNAI, PUNE, BENGALURU, Hyderabad Mode of Interview : Virtual Date of Interview

In [ ]:
## MMR 

lambda_values = [1.0, 0.7, 0.5 ,0.0]

for lm in lambda_values:
    retriever = vector_store.as_retriever(
        search_type="mmr",
        search_kwargs={"k": 3, "fetch_k": 10, "lambda_mult": lm},
    )
    results = retriever.invoke(query)
    
    topics = [doc.metadata for doc in results]
    print(f"=== lambda_mult={lm} ===")
    for i, doc in enumerate(results, 1):
        print(f"  [{i}] topic={doc.metadata}: {doc.page_content}")
    print()

=== lambda_mult=1.0 ===
  [1] topic={'Description': 'innodata india job description', 'source': 'innodata_india_de'}: Job description Role- GCP Data Engineer Responsibilities: • Design and implement data-driven solutions on GCP including BigQuery, Cloud Storage, Dataflow, Pub/Sub, and Looker/BI. • Build ETL scripts using SQL and Python to extract, clean, and transform structured and unstructured data from ERP, procurement, logistics, and facility management systems. • Develop and optimize data pipelines for ingestion, transformation, and loading into enterprise data lakes and warehouses. • Build and extend
  [2] topic={'source': 'tcs_gcp_de', 'Description': 'tcs gcp job description'}: Job description TCS Hiring for Digital : GCP Data Engineer - BigQuery Role!! TCS presents an excellent opportunity forGCP Data Engineer - BigQuery Desired Experience Range: 5-10 years Location: CHENNAI, PUNE, BENGALURU, Hyderabad Mode of Interview : Virtual Date of Interview : 14-05-26 Desired Competencie

In [38]:
bm_retriever = BM25Retriever.from_documents(
    documents=all_documents
)

bm_retriever.k = 3

bm_results = bm_retriever.invoke(
    query
)

bm_results

[Document(metadata={'source': 'innodata_india_de', 'Description': 'innodata india job description'}, page_content='to data pipelines for AI/ML (vector DB ingestion, embeddings, RAG pipelines, copilots, agents). • Familiarity with supply chain or data center operations data is a strong plus. Role: Data Engineer Industry Type: IT Services & Consulting Department: Engineering - Software & QA Employment Type: Full Time, Permanent Role Category: Software Development Education UG: B.Tech / B.E. in Any Specialization Key Skills Skills highlighted with ‘‘ are preferred keyskills ETL/ELT GCP Bigquery Cloud Storage'),
 Document(metadata={'source': 'visa_de', 'Description': 'visa de job description'}, page_content='highlighted with ‘‘ are preferred keyskills Data Engineer Data Engineering Data Pipelines Architecture Power Bi Data Warehouse Technical Leadership Version Control Tableau Analytics Hive Data Quality Git Spark Data Modeling Etl Policy Development Data Storage Python'),
 Document(metada

In [39]:
chroma_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5},
)

ensemble_retriever = EnsembleRetriever(
    retrievers=[chroma_retriever, bm_retriever],
    weights=[0.8, 0.2]
)

ensemble_results = ensemble_retriever.invoke(
    query
)

for i, doc in enumerate(ensemble_results, 1):
    print(f"  [{i}] {doc.metadata}: {doc.page_content}")

  [1] {'source': 'innodata_india_de', 'Description': 'innodata india job description'}: Job description Role- GCP Data Engineer Responsibilities: • Design and implement data-driven solutions on GCP including BigQuery, Cloud Storage, Dataflow, Pub/Sub, and Looker/BI. • Build ETL scripts using SQL and Python to extract, clean, and transform structured and unstructured data from ERP, procurement, logistics, and facility management systems. • Develop and optimize data pipelines for ingestion, transformation, and loading into enterprise data lakes and warehouses. • Build and extend
  [2] {'Description': 'tcs gcp job description', 'source': 'tcs_gcp_de'}: Job description TCS Hiring for Digital : GCP Data Engineer - BigQuery Role!! TCS presents an excellent opportunity forGCP Data Engineer - BigQuery Desired Experience Range: 5-10 years Location: CHENNAI, PUNE, BENGALURU, Hyderabad Mode of Interview : Virtual Date of Interview : 14-05-26 Desired Competencies: (Technical/Behavioural Competency

In [59]:
llm = OpenAI(temperature=0)

compressor = LLMChainExtractor.from_llm(
    llm=llm
)

compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=retriever
)

compressed_docs = compression_retriever.invoke(query)

lambda_values = [0.0]

for lm in lambda_values:
    retriever = vector_store.as_retriever(
        search_type="mmr",
        search_kwargs={"k": 3, "fetch_k": 10, "lambda_mult": lm},
    )
    results = retriever.invoke(query)
    
    topics = [doc.metadata for doc in results]
    print(f"=== lambda_mult={lm} ===")
    for i, doc in enumerate(results, 1):
        print(len(doc.page_content))
        print(f"  [{i}] metadata={doc.metadata}: {doc.page_content}")
    print()

print("=" * 50)

print(len(compressed_docs[0].page_content))
print(compressed_docs)

=== lambda_mult=0.0 ===
495
  [1] metadata={'source': 'innodata_india_de', 'Description': 'innodata india job description'}: Job description Role- GCP Data Engineer Responsibilities: • Design and implement data-driven solutions on GCP including BigQuery, Cloud Storage, Dataflow, Pub/Sub, and Looker/BI. • Build ETL scripts using SQL and Python to extract, clean, and transform structured and unstructured data from ERP, procurement, logistics, and facility management systems. • Develop and optimize data pipelines for ingestion, transformation, and loading into enterprise data lakes and warehouses. • Build and extend
497
  [2] metadata={'source': 'accenture_data_platform_de', 'Description': 'accenture data job description'}: Job description Project Role :Data Platform Engineer Project Role Description :Assists with the data platform blueprint and design, encompassing the relevant data platform components. Collaborates with the Integration Architects and Data Architects to ensure cohesive i